Copyright © 2025, Andrea Mastropietro. All rights reserved.

This code is licensed under the MIT License.

See the LICENSE file in the project root for more information.

## Analysis of anchor and non-anchor atoms using a control based on E(3)-transformations (distance-based vs Coulomb-matrix based)

In [1]:
import os
# os.environ["http_proxy"] = "http://web-proxy.informatik.uni-bonn.de:3128"
# os.environ["https_proxy"] = "http://web-proxy.informatik.uni-bonn.de:3128"

import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [2]:
# Standard library imports
import copy
import os

# Third-party imports
import yaml
import torch
import numpy as np
from scipy.stats import spearmanr, kendalltau, pearsonr

# Visualization imports
import matplotlib.pyplot as plt
import seaborn as sns

# Project-specific imports
from src.difflinker.datasets import get_dataloader
from src.difflinker.lightning import DDPM

In [3]:
with open('../config.yml', 'r') as file:
    config = yaml.safe_load(file)

checkpoint = "../" + config['CHECKPOINT']
DATA_FOLDER = "../" + config['DATA_FOLDER']
DATASET_NAME = config['DATASET_NAME']
device = config['DEVICE'] if torch.cuda.is_available() else 'cpu'
NUM_SAMPLES = config['NUM_SAMPLES']
P = config['P']
ATOM_TYPE_PERTURBATION = config['ATOM_TYPE_PERTURBATION']

SAVE_PLOT_FOLDER = "../results/plots/frequency_analysis/"
SHAPLEY_VALUE_FOLDER = f"../results/explanations/{DATASET_NAME}/" 
SHAPLEY_VALUE_FOLDER_CHAMFER = f"../results/chamfer_distance/explanations/{DATASET_NAME}/"
SHAPLEY_VALUE_FOLDER_RMSD = f"../results/rmsd/explanations/{DATASET_NAME}/"
POSITIONS_FOLDER = f"../results/explanations/{DATASET_NAME}/" 

if ATOM_TYPE_PERTURBATION:
    SAVE_PLOT_FOLDER += "including_atom_type_perturbation/"
    
os.makedirs(SAVE_PLOT_FOLDER, exist_ok=True)

Load model to load the data

In [4]:
model = DDPM.load_from_checkpoint(checkpoint, map_location=device)
model.val_data_prefix = DATASET_NAME

print(f"Running device: {device}")

model.data_path = DATA_FOLDER

model = model.eval().to(device)
model.setup(stage='val')
dataloader = get_dataloader(
    model.val_dataset,
    batch_size=1
)

c:\Users\Mastro\anaconda3\envs\diff_explainer\lib\site-packages\lightning_fabric\utilities\cloud_io.py:57: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
Lightning automatically upgraded your

Running device: cuda:0


c:\Repositories\DiffSHAPer\src\difflinker\datasets.py:46: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data = torch.load(dataset_path, map_location=device)


Load data samples for fragment, linker and anchor indices

In [5]:
data_list = []
sampled = 0
data_dict = {}
for data in dataloader:
    if sampled < NUM_SAMPLES:
        data_list.append(data)
        sampled += 1

#print all fragment masks, linker masks, and anchor masks
for i in range(len(data_list)):
    data_dict[i] = {}
    data_dict[i]["fragment_mask"] = data_list[i]['fragment_mask'].squeeze(0)
    data_dict[i]["linker_mask"] = data_list[i]['linker_mask'].squeeze(0)
    data_dict[i]["anchors"] = data_list[i]['anchors'].squeeze(0)


### Analyzing distance-based Shapley values

Read Shapley values from disk

In [6]:
seed_list = [42]
# STRATEGY = "hausdorff_distance"
for seed in seed_list:
    for i in range(NUM_SAMPLES):
        with open(f"{SHAPLEY_VALUE_FOLDER}explanations_seed_{str(seed)}/shapley_values/shapley_values_atoms_{i}.txt", "r") as f:
            f.readline()
            f.readline()
            shapley_values = []
            for line in f:
                if line == "\n":
                    break
                row = line.strip().split(",")
                shapley_values.append(float(row[1]))
            dict_key_name = f"shapley_values_{seed}"
            data_dict[i][dict_key_name] = shapley_values


read final positions from disk

In [7]:
for seed in seed_list:
    for i in range(NUM_SAMPLES):
        with open(f"{POSITIONS_FOLDER}explanations_seed_{str(seed)}/mapping/graphs/{i}/{i}_0_.xyz", "r") as f: #file with final atom positions
            num_atoms = int(f.readline().strip())
            f.readline()
            positions = []*num_atoms
            for line in f:
                row = line.strip().split(" ")[1:]
                coords = [float(x) for x in row]
                positions.append(coords)
            dict_key_name = f"positions_{seed}"
            data_dict[i][dict_key_name] = positions

Compute center of mass of linker atoms

In [8]:
for i in range(len(data_list)):
    linker_mask = data_dict[i]['linker_mask'].bool().cpu().numpy()
    for seed in seed_list:
        dict_key_name = f"positions_{seed}"
        positions = np.array(data_dict[i][dict_key_name])
        linker_positions = positions[linker_mask.squeeze()]
        com_linker = np.mean(linker_positions, axis=0)
        dict_key_name = f"com_linker_{seed}"
        data_dict[i][dict_key_name] = com_linker

Compute distance between fragment atoms and linker COM

In [9]:
for i in range(len(data_list)):
    fragment_mask = data_dict[i]['fragment_mask'].bool().cpu().numpy()
    for seed in seed_list:
        fragment_positions = np.array(data_dict[i][f'positions_{seed}'])[fragment_mask.squeeze()]
        com_linker = data_dict[i][f'com_linker_{seed}']
        distances = np.linalg.norm(fragment_positions - com_linker, axis=1)
        dict_key_name = f'distances_{seed}'
        data_dict[i][dict_key_name] = distances

Boxplot generation

In [10]:
for i in range(len(data_list)):
    for seed in seed_list:
        dict_key_name = f"shapley_values_{seed}"
        shapley_values = -np.array(data_dict[i][dict_key_name])
        min_val = np.min(shapley_values)
        max_val = np.max(shapley_values)
        normalized_shapley_values = 2 * (shapley_values - min_val) / (max_val - min_val) - 1
        dict_key_name = f"normalized_shapley_values_{seed}"
        data_dict[i][dict_key_name] = normalized_shapley_values.tolist()

In [11]:
for i in range(len(data_list)):
    for seed in seed_list:
        dict_key_name = f'distances_{seed}'
        distances = np.array(data_dict[i][dict_key_name])
        min_dist = np.min(distances)
        max_dist = np.max(distances)
        normalized_distances = (distances - min_dist) / (max_dist - min_dist)
        dict_key_name = f'normalized_distances_{seed}'
        data_dict[i][dict_key_name] = normalized_distances.tolist()

In [12]:
# Read and normalize Shapley values for Chamfer distance and RMSD
# (same normalization style as before: negate, then scale to [-1, 1])

seed_list = [42]
# STRATEGY = "hausdorff_distance"
for seed in seed_list:
    for i in range(NUM_SAMPLES):
        with open(f"{SHAPLEY_VALUE_FOLDER_CHAMFER}explanations_seed_{str(seed)}/shapley_values/shapley_values_atoms_{i}.txt", "r") as f:
            f.readline()
            f.readline()
            shapley_values = []
            for line in f:
                if line == "\n":
                    break
                row = line.strip().split(",")
                shapley_values.append(float(row[1]))
            dict_key_name = f"shapley_values_chamfer_{seed}"
            data_dict[i][dict_key_name] = shapley_values


for i in range(len(data_list)):
    for seed in seed_list:
        dict_key_name = f"shapley_values_chamfer_{seed}"
        shapley_values = -np.array(data_dict[i][dict_key_name])
        min_val = np.min(shapley_values)
        max_val = np.max(shapley_values)
        normalized_shapley_values = 2 * (shapley_values - min_val) / (max_val - min_val) - 1
        dict_key_name = f"normalized_shapley_values_chamfer_{seed}"
        data_dict[i][dict_key_name] = normalized_shapley_values.tolist()



for seed in seed_list:
    for i in range(NUM_SAMPLES):
        with open(f"{SHAPLEY_VALUE_FOLDER_RMSD}explanations_seed_{str(seed)}/shapley_values/shapley_values_atoms_{i}.txt", "r") as f:
            f.readline()
            f.readline()
            shapley_values = []
            for line in f:
                if line == "\n":
                    break
                row = line.strip().split(",")
                shapley_values.append(float(row[1]))
            dict_key_name = f"shapley_values_rmsd_{seed}"
            data_dict[i][dict_key_name] = shapley_values


for i in range(len(data_list)):
    for seed in seed_list:
        dict_key_name = f"shapley_values_rmsd_{seed}"
        shapley_values = -np.array(data_dict[i][dict_key_name])
        min_val = np.min(shapley_values)
        max_val = np.max(shapley_values)
        normalized_shapley_values = 2 * (shapley_values - min_val) / (max_val - min_val) - 1
        dict_key_name = f"normalized_shapley_values_rmsd_{seed}"
        data_dict[i][dict_key_name] = normalized_shapley_values.tolist()

In [14]:
# Frequency analysis for anchor atoms and their fragment-neighbors in top-k Shapley atoms
seed = 42 if "seed" not in globals() else seed
top_k = 5 if "top_k" not in globals() else top_k
neighbor_k = 2  # number of nearest fragment neighbors per anchor

anchor_neighbor_topk_stats = {}

tot_anchor_hits = tot_neighbor_hits = tot_combined_hits = 0
tot_topk_slots = 0
tot_anchor_atoms = tot_neighbor_atoms = tot_combined_atoms = 0
samples_anchor_any = samples_neighbor_any = samples_combined_any = 0
valid_samples = 0

for i in range(len(data_list)):
    shapley = np.array(data_dict[i][f"normalized_shapley_values_{seed}"])
    if shapley.size == 0:
        continue

    fragment_mask = data_dict[i]["fragment_mask"].bool().cpu().numpy().squeeze()
    n_atoms = fragment_mask.shape[0]
    fragment_global_idx = np.where(fragment_mask)[0]

    anchors_raw = data_dict[i]["anchors"].cpu().numpy().squeeze()
    anchors_flat = np.array(anchors_raw).reshape(-1)

    # anchor extraction (mask-style or index-style)
    if anchors_flat.size == n_atoms and np.all(np.isin(anchors_flat, [0, 1])):
        anchor_global_idx = np.where(anchors_flat > 0)[0]
    else:
        anchor_global_idx = anchors_flat.astype(int)
        anchor_global_idx = anchor_global_idx[(anchor_global_idx >= 0) & (anchor_global_idx < n_atoms)]
        anchor_global_idx = np.unique(anchor_global_idx)

    # keep only anchors inside fragment atoms
    anchor_global_idx = np.array([a for a in anchor_global_idx if a in set(fragment_global_idx)], dtype=int)

    # top-k indices in shapley space
    k = min(top_k, shapley.size)
    topk_idx = np.argsort(shapley)[::-1][:k]

    # map global atom index -> shapley index space
    if shapley.size == fragment_global_idx.size:
        global_to_shapley = {g: l for l, g in enumerate(fragment_global_idx)}
    else:
        global_to_shapley = {g: g for g in range(min(n_atoms, shapley.size))}

    anchor_idx = np.array([global_to_shapley[g] for g in anchor_global_idx if g in global_to_shapley], dtype=int)

    # nearest fragment neighbors of anchors from coordinates
    positions_key = f"positions_{seed}"
    if positions_key not in data_dict[i]:
        continue
    pos = np.array(data_dict[i][positions_key])
    frag_pos = pos[fragment_global_idx]

    frag_global_to_local = {g: l for l, g in enumerate(fragment_global_idx)}
    anchor_local = np.array([frag_global_to_local[g] for g in anchor_global_idx if g in frag_global_to_local], dtype=int)

    neighbor_global_set = set()
    if anchor_local.size > 0:
        for a_local in anchor_local:
            d = np.linalg.norm(frag_pos - frag_pos[a_local], axis=1)
            order = np.argsort(d)
            # skip itself and pick nearest non-anchor fragment atoms
            picked = 0
            for idx_local in order:
                g_idx = fragment_global_idx[idx_local]
                if g_idx == fragment_global_idx[a_local] or g_idx in set(anchor_global_idx):
                    continue
                neighbor_global_set.add(int(g_idx))
                picked += 1
                if picked >= neighbor_k:
                    break

    neighbor_idx = np.array(
        [global_to_shapley[g] for g in sorted(neighbor_global_set) if g in global_to_shapley],
        dtype=int
    )

    combined_idx = np.unique(np.concatenate([anchor_idx, neighbor_idx])) if (anchor_idx.size + neighbor_idx.size) > 0 else np.array([], dtype=int)

    anchor_hits = np.intersect1d(topk_idx, anchor_idx).size
    neighbor_hits = np.intersect1d(topk_idx, neighbor_idx).size
    combined_hits = np.intersect1d(topk_idx, combined_idx).size

    tot_anchor_hits += anchor_hits
    tot_neighbor_hits += neighbor_hits
    tot_combined_hits += combined_hits
    tot_topk_slots += k

    tot_anchor_atoms += anchor_idx.size
    tot_neighbor_atoms += neighbor_idx.size
    tot_combined_atoms += combined_idx.size

    samples_anchor_any += int(anchor_hits > 0)
    samples_neighbor_any += int(neighbor_hits > 0)
    samples_combined_any += int(combined_hits > 0)
    valid_samples += 1

anchor_neighbor_topk_stats[seed] = {
    "anchor_hit_rate_in_topk_slots": (tot_anchor_hits / tot_topk_slots) if tot_topk_slots > 0 else np.nan,
    "neighbor_hit_rate_in_topk_slots": (tot_neighbor_hits / tot_topk_slots) if tot_topk_slots > 0 else np.nan,
    "anchor_or_neighbor_hit_rate_in_topk_slots": (tot_combined_hits / tot_topk_slots) if tot_topk_slots > 0 else np.nan,
    "anchor_coverage_rate": (tot_anchor_hits / tot_anchor_atoms) if tot_anchor_atoms > 0 else np.nan,
    "neighbor_coverage_rate": (tot_neighbor_hits / tot_neighbor_atoms) if tot_neighbor_atoms > 0 else np.nan,
    "anchor_or_neighbor_coverage_rate": (tot_combined_hits / tot_combined_atoms) if tot_combined_atoms > 0 else np.nan,
    "sample_freq_any_anchor_in_topk": (samples_anchor_any / valid_samples) if valid_samples > 0 else np.nan,
    "sample_freq_any_neighbor_in_topk": (samples_neighbor_any / valid_samples) if valid_samples > 0 else np.nan,
    "sample_freq_any_anchor_or_neighbor_in_topk": (samples_combined_any / valid_samples) if valid_samples > 0 else np.nan,
    "valid_samples": valid_samples,
    "neighbor_k": neighbor_k,
    "top_k": top_k,
}

stats_an = anchor_neighbor_topk_stats[seed]
print(f"Seed {seed} | top_k={top_k}, neighbor_k={neighbor_k}")
for k, v in stats_an.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

Seed 42 | top_k=5, neighbor_k=2
  anchor_hit_rate_in_topk_slots: 0.2200
  neighbor_hit_rate_in_topk_slots: 0.2933
  anchor_or_neighbor_hit_rate_in_topk_slots: 0.5133
  anchor_coverage_rate: 0.5500
  neighbor_coverage_rate: 0.3667
  anchor_or_neighbor_coverage_rate: 0.4278
  sample_freq_any_anchor_in_topk: 0.7667
  sample_freq_any_neighbor_in_topk: 0.8667
  sample_freq_any_anchor_or_neighbor_in_topk: 0.9667
  valid_samples: 30
  neighbor_k: 2
  top_k: 5


Atom frequency for Chamfer and RMSD

In [15]:
# Frequency analysis for anchor atoms and their fragment-neighbors in top-k Shapley atoms
seed = 42 if "seed" not in globals() else seed
top_k = 5 if "top_k" not in globals() else top_k
neighbor_k = 2  # number of nearest fragment neighbors per anchor

anchor_neighbor_topk_stats = {}

tot_anchor_hits = tot_neighbor_hits = tot_combined_hits = 0
tot_topk_slots = 0
tot_anchor_atoms = tot_neighbor_atoms = tot_combined_atoms = 0
samples_anchor_any = samples_neighbor_any = samples_combined_any = 0
valid_samples = 0

for i in range(len(data_list)):
    shapley = np.array(data_dict[i][f"normalized_shapley_values_chamfer_{seed}"])
    if shapley.size == 0:
        continue

    fragment_mask = data_dict[i]["fragment_mask"].bool().cpu().numpy().squeeze()
    n_atoms = fragment_mask.shape[0]
    fragment_global_idx = np.where(fragment_mask)[0]

    anchors_raw = data_dict[i]["anchors"].cpu().numpy().squeeze()
    anchors_flat = np.array(anchors_raw).reshape(-1)

    # anchor extraction (mask-style or index-style)
    if anchors_flat.size == n_atoms and np.all(np.isin(anchors_flat, [0, 1])):
        anchor_global_idx = np.where(anchors_flat > 0)[0]
    else:
        anchor_global_idx = anchors_flat.astype(int)
        anchor_global_idx = anchor_global_idx[(anchor_global_idx >= 0) & (anchor_global_idx < n_atoms)]
        anchor_global_idx = np.unique(anchor_global_idx)

    # keep only anchors inside fragment atoms
    anchor_global_idx = np.array([a for a in anchor_global_idx if a in set(fragment_global_idx)], dtype=int)

    # top-k indices in shapley space
    k = min(top_k, shapley.size)
    topk_idx = np.argsort(shapley)[::-1][:k]

    # map global atom index -> shapley index space
    if shapley.size == fragment_global_idx.size:
        global_to_shapley = {g: l for l, g in enumerate(fragment_global_idx)}
    else:
        global_to_shapley = {g: g for g in range(min(n_atoms, shapley.size))}

    anchor_idx = np.array([global_to_shapley[g] for g in anchor_global_idx if g in global_to_shapley], dtype=int)

    # nearest fragment neighbors of anchors from coordinates
    positions_key = f"positions_{seed}"
    if positions_key not in data_dict[i]:
        continue
    pos = np.array(data_dict[i][positions_key])
    frag_pos = pos[fragment_global_idx]

    frag_global_to_local = {g: l for l, g in enumerate(fragment_global_idx)}
    anchor_local = np.array([frag_global_to_local[g] for g in anchor_global_idx if g in frag_global_to_local], dtype=int)

    neighbor_global_set = set()
    if anchor_local.size > 0:
        for a_local in anchor_local:
            d = np.linalg.norm(frag_pos - frag_pos[a_local], axis=1)
            order = np.argsort(d)
            # skip itself and pick nearest non-anchor fragment atoms
            picked = 0
            for idx_local in order:
                g_idx = fragment_global_idx[idx_local]
                if g_idx == fragment_global_idx[a_local] or g_idx in set(anchor_global_idx):
                    continue
                neighbor_global_set.add(int(g_idx))
                picked += 1
                if picked >= neighbor_k:
                    break

    neighbor_idx = np.array(
        [global_to_shapley[g] for g in sorted(neighbor_global_set) if g in global_to_shapley],
        dtype=int
    )

    combined_idx = np.unique(np.concatenate([anchor_idx, neighbor_idx])) if (anchor_idx.size + neighbor_idx.size) > 0 else np.array([], dtype=int)

    anchor_hits = np.intersect1d(topk_idx, anchor_idx).size
    neighbor_hits = np.intersect1d(topk_idx, neighbor_idx).size
    combined_hits = np.intersect1d(topk_idx, combined_idx).size

    tot_anchor_hits += anchor_hits
    tot_neighbor_hits += neighbor_hits
    tot_combined_hits += combined_hits
    tot_topk_slots += k

    tot_anchor_atoms += anchor_idx.size
    tot_neighbor_atoms += neighbor_idx.size
    tot_combined_atoms += combined_idx.size

    samples_anchor_any += int(anchor_hits > 0)
    samples_neighbor_any += int(neighbor_hits > 0)
    samples_combined_any += int(combined_hits > 0)
    valid_samples += 1

anchor_neighbor_topk_stats[seed] = {
    "anchor_hit_rate_in_topk_slots": (tot_anchor_hits / tot_topk_slots) if tot_topk_slots > 0 else np.nan,
    "neighbor_hit_rate_in_topk_slots": (tot_neighbor_hits / tot_topk_slots) if tot_topk_slots > 0 else np.nan,
    "anchor_or_neighbor_hit_rate_in_topk_slots": (tot_combined_hits / tot_topk_slots) if tot_topk_slots > 0 else np.nan,
    "anchor_coverage_rate": (tot_anchor_hits / tot_anchor_atoms) if tot_anchor_atoms > 0 else np.nan,
    "neighbor_coverage_rate": (tot_neighbor_hits / tot_neighbor_atoms) if tot_neighbor_atoms > 0 else np.nan,
    "anchor_or_neighbor_coverage_rate": (tot_combined_hits / tot_combined_atoms) if tot_combined_atoms > 0 else np.nan,
    "sample_freq_any_anchor_in_topk": (samples_anchor_any / valid_samples) if valid_samples > 0 else np.nan,
    "sample_freq_any_neighbor_in_topk": (samples_neighbor_any / valid_samples) if valid_samples > 0 else np.nan,
    "sample_freq_any_anchor_or_neighbor_in_topk": (samples_combined_any / valid_samples) if valid_samples > 0 else np.nan,
    "valid_samples": valid_samples,
    "neighbor_k": neighbor_k,
    "top_k": top_k,
}

stats_an = anchor_neighbor_topk_stats[seed]
print(f"Seed {seed} | top_k={top_k}, neighbor_k={neighbor_k}")
for k, v in stats_an.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

Seed 42 | top_k=5, neighbor_k=2
  anchor_hit_rate_in_topk_slots: 0.2133
  neighbor_hit_rate_in_topk_slots: 0.3133
  anchor_or_neighbor_hit_rate_in_topk_slots: 0.5267
  anchor_coverage_rate: 0.5333
  neighbor_coverage_rate: 0.3917
  anchor_or_neighbor_coverage_rate: 0.4389
  sample_freq_any_anchor_in_topk: 0.7333
  sample_freq_any_neighbor_in_topk: 0.8667
  sample_freq_any_anchor_or_neighbor_in_topk: 0.9667
  valid_samples: 30
  neighbor_k: 2
  top_k: 5


In [16]:
# Frequency analysis for anchor atoms and their fragment-neighbors in top-k Shapley atoms
seed = 42 if "seed" not in globals() else seed
top_k = 5 if "top_k" not in globals() else top_k
neighbor_k = 2  # number of nearest fragment neighbors per anchor

anchor_neighbor_topk_stats = {}

tot_anchor_hits = tot_neighbor_hits = tot_combined_hits = 0
tot_topk_slots = 0
tot_anchor_atoms = tot_neighbor_atoms = tot_combined_atoms = 0
samples_anchor_any = samples_neighbor_any = samples_combined_any = 0
valid_samples = 0

for i in range(len(data_list)):
    shapley = np.array(data_dict[i][f"normalized_shapley_values_rmsd_{seed}"])
    if shapley.size == 0:
        continue

    fragment_mask = data_dict[i]["fragment_mask"].bool().cpu().numpy().squeeze()
    n_atoms = fragment_mask.shape[0]
    fragment_global_idx = np.where(fragment_mask)[0]

    anchors_raw = data_dict[i]["anchors"].cpu().numpy().squeeze()
    anchors_flat = np.array(anchors_raw).reshape(-1)

    # anchor extraction (mask-style or index-style)
    if anchors_flat.size == n_atoms and np.all(np.isin(anchors_flat, [0, 1])):
        anchor_global_idx = np.where(anchors_flat > 0)[0]
    else:
        anchor_global_idx = anchors_flat.astype(int)
        anchor_global_idx = anchor_global_idx[(anchor_global_idx >= 0) & (anchor_global_idx < n_atoms)]
        anchor_global_idx = np.unique(anchor_global_idx)

    # keep only anchors inside fragment atoms
    anchor_global_idx = np.array([a for a in anchor_global_idx if a in set(fragment_global_idx)], dtype=int)

    # top-k indices in shapley space
    k = min(top_k, shapley.size)
    topk_idx = np.argsort(shapley)[::-1][:k]

    # map global atom index -> shapley index space
    if shapley.size == fragment_global_idx.size:
        global_to_shapley = {g: l for l, g in enumerate(fragment_global_idx)}
    else:
        global_to_shapley = {g: g for g in range(min(n_atoms, shapley.size))}

    anchor_idx = np.array([global_to_shapley[g] for g in anchor_global_idx if g in global_to_shapley], dtype=int)

    # nearest fragment neighbors of anchors from coordinates
    positions_key = f"positions_{seed}"
    if positions_key not in data_dict[i]:
        continue
    pos = np.array(data_dict[i][positions_key])
    frag_pos = pos[fragment_global_idx]

    frag_global_to_local = {g: l for l, g in enumerate(fragment_global_idx)}
    anchor_local = np.array([frag_global_to_local[g] for g in anchor_global_idx if g in frag_global_to_local], dtype=int)

    neighbor_global_set = set()
    if anchor_local.size > 0:
        for a_local in anchor_local:
            d = np.linalg.norm(frag_pos - frag_pos[a_local], axis=1)
            order = np.argsort(d)
            # skip itself and pick nearest non-anchor fragment atoms
            picked = 0
            for idx_local in order:
                g_idx = fragment_global_idx[idx_local]
                if g_idx == fragment_global_idx[a_local] or g_idx in set(anchor_global_idx):
                    continue
                neighbor_global_set.add(int(g_idx))
                picked += 1
                if picked >= neighbor_k:
                    break

    neighbor_idx = np.array(
        [global_to_shapley[g] for g in sorted(neighbor_global_set) if g in global_to_shapley],
        dtype=int
    )

    combined_idx = np.unique(np.concatenate([anchor_idx, neighbor_idx])) if (anchor_idx.size + neighbor_idx.size) > 0 else np.array([], dtype=int)

    anchor_hits = np.intersect1d(topk_idx, anchor_idx).size
    neighbor_hits = np.intersect1d(topk_idx, neighbor_idx).size
    combined_hits = np.intersect1d(topk_idx, combined_idx).size

    tot_anchor_hits += anchor_hits
    tot_neighbor_hits += neighbor_hits
    tot_combined_hits += combined_hits
    tot_topk_slots += k

    tot_anchor_atoms += anchor_idx.size
    tot_neighbor_atoms += neighbor_idx.size
    tot_combined_atoms += combined_idx.size

    samples_anchor_any += int(anchor_hits > 0)
    samples_neighbor_any += int(neighbor_hits > 0)
    samples_combined_any += int(combined_hits > 0)
    valid_samples += 1

anchor_neighbor_topk_stats[seed] = {
    "anchor_hit_rate_in_topk_slots": (tot_anchor_hits / tot_topk_slots) if tot_topk_slots > 0 else np.nan,
    "neighbor_hit_rate_in_topk_slots": (tot_neighbor_hits / tot_topk_slots) if tot_topk_slots > 0 else np.nan,
    "anchor_or_neighbor_hit_rate_in_topk_slots": (tot_combined_hits / tot_topk_slots) if tot_topk_slots > 0 else np.nan,
    "anchor_coverage_rate": (tot_anchor_hits / tot_anchor_atoms) if tot_anchor_atoms > 0 else np.nan,
    "neighbor_coverage_rate": (tot_neighbor_hits / tot_neighbor_atoms) if tot_neighbor_atoms > 0 else np.nan,
    "anchor_or_neighbor_coverage_rate": (tot_combined_hits / tot_combined_atoms) if tot_combined_atoms > 0 else np.nan,
    "sample_freq_any_anchor_in_topk": (samples_anchor_any / valid_samples) if valid_samples > 0 else np.nan,
    "sample_freq_any_neighbor_in_topk": (samples_neighbor_any / valid_samples) if valid_samples > 0 else np.nan,
    "sample_freq_any_anchor_or_neighbor_in_topk": (samples_combined_any / valid_samples) if valid_samples > 0 else np.nan,
    "valid_samples": valid_samples,
    "neighbor_k": neighbor_k,
    "top_k": top_k,
}

stats_an = anchor_neighbor_topk_stats[seed]
print(f"Seed {seed} | top_k={top_k}, neighbor_k={neighbor_k}")
for k, v in stats_an.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

Seed 42 | top_k=5, neighbor_k=2
  anchor_hit_rate_in_topk_slots: 0.2000
  neighbor_hit_rate_in_topk_slots: 0.2800
  anchor_or_neighbor_hit_rate_in_topk_slots: 0.4800
  anchor_coverage_rate: 0.5000
  neighbor_coverage_rate: 0.3500
  anchor_or_neighbor_coverage_rate: 0.4000
  sample_freq_any_anchor_in_topk: 0.7333
  sample_freq_any_neighbor_in_topk: 0.8333
  sample_freq_any_anchor_or_neighbor_in_topk: 0.9667
  valid_samples: 30
  neighbor_k: 2
  top_k: 5


Monte Carlo permutation (randomization) test with a one-sided alternative to build a null distribution - test against randomicity of importance ranking

In [17]:
# Permutation test (ANCHORS ONLY): observed top-k vs random top-k
n_perm = 20000
rng = np.random.default_rng(42)

obs_anchor_hits = 0
obs_topk_slots = 0
obs_samples_any_anchor = 0
valid_samples = 0

# Cache per-sample index sets once
cache = []

for i in range(len(data_list)):
    s_key = f"normalized_shapley_values_{seed}"
    if s_key not in data_dict[i]:
        continue
    shapley = np.array(data_dict[i][s_key])
    if shapley.size == 0:
        continue

    fragment_mask = data_dict[i]["fragment_mask"].bool().cpu().numpy().squeeze()
    n_atoms = fragment_mask.shape[0]
    fragment_global_idx = np.where(fragment_mask)[0]

    anchors_raw = data_dict[i]["anchors"].cpu().numpy().squeeze()
    anchors_flat = np.array(anchors_raw).reshape(-1)

    # anchor extraction
    if anchors_flat.size == n_atoms and np.all(np.isin(anchors_flat, [0, 1])):
        anchor_global_idx = np.where(anchors_flat > 0)[0]
    else:
        anchor_global_idx = anchors_flat.astype(int)
        anchor_global_idx = anchor_global_idx[(anchor_global_idx >= 0) & (anchor_global_idx < n_atoms)]
        anchor_global_idx = np.unique(anchor_global_idx)

    # keep only anchors inside fragment atoms
    frag_set = set(fragment_global_idx.tolist())
    anchor_global_idx = np.array([a for a in anchor_global_idx if a in frag_set], dtype=int)

    k = min(top_k, shapley.size)
    topk_obs = np.argsort(shapley)[::-1][:k]

    # map global atom index -> shapley index space
    if shapley.size == fragment_global_idx.size:
        global_to_shapley = {g: l for l, g in enumerate(fragment_global_idx)}
    else:
        global_to_shapley = {g: g for g in range(min(n_atoms, shapley.size))}

    anchor_idx = np.array([global_to_shapley[g] for g in anchor_global_idx if g in global_to_shapley], dtype=int)

    # observed (anchors only)
    anchor_hits = np.intersect1d(topk_obs, anchor_idx).size
    obs_anchor_hits += anchor_hits
    obs_topk_slots += k
    obs_samples_any_anchor += int(anchor_hits > 0)
    valid_samples += 1

    cache.append((shapley.size, k, anchor_idx))

obs_hit_rate = obs_anchor_hits / obs_topk_slots if obs_topk_slots > 0 else np.nan
obs_sample_freq = obs_samples_any_anchor / valid_samples if valid_samples > 0 else np.nan

# null distribution
null_hit_rates = np.zeros(n_perm, dtype=float)
null_sample_freqs = np.zeros(n_perm, dtype=float)

for p in range(n_perm):
    rnd_hits = 0
    rnd_slots = 0
    rnd_any = 0
    for N, k, anchor_idx in cache:
        rnd_topk = rng.choice(N, size=k, replace=False)
        h = np.intersect1d(rnd_topk, anchor_idx).size
        rnd_hits += h
        rnd_slots += k
        rnd_any += int(h > 0)
    null_hit_rates[p] = rnd_hits / rnd_slots if rnd_slots > 0 else np.nan
    null_sample_freqs[p] = rnd_any / len(cache) if len(cache) > 0 else np.nan

# one-sided empirical p-values: observed > random
p_hit = (1 + np.sum(null_hit_rates >= obs_hit_rate)) / (n_perm + 1)
p_freq = (1 + np.sum(null_sample_freqs >= obs_sample_freq)) / (n_perm + 1)

print(f"Observed anchor_hit_rate_in_topk_slots: {obs_hit_rate:.4f}")
print(f"Random mean±std: {np.nanmean(null_hit_rates):.4f} ± {np.nanstd(null_hit_rates, ddof=1):.4f}")
print(f"Empirical p-value (hit rate): {p_hit:.4g}")

print(f"Observed sample_freq_any_anchor_in_topk: {obs_sample_freq:.4f}")
print(f"Random mean±std: {np.nanmean(null_sample_freqs):.4f} ± {np.nanstd(null_sample_freqs, ddof=1):.4f}")
print(f"Empirical p-value (sample freq): {p_freq:.4g}")

# Effect sizes
z_hit = (obs_hit_rate - np.nanmean(null_hit_rates)) / (np.nanstd(null_hit_rates, ddof=1) + 1e-12)
z_freq = (obs_sample_freq - np.nanmean(null_sample_freqs)) / (np.nanstd(null_sample_freqs, ddof=1) + 1e-12)

# 95% null intervals
ci_hit = np.nanpercentile(null_hit_rates, [2.5, 97.5])
ci_freq = np.nanpercentile(null_sample_freqs, [2.5, 97.5])

print(f"Z-score (hit rate): {z_hit:.2f}")
print(f"Z-score (sample freq): {z_freq:.2f}")
print(f"Null 95% interval (hit rate): [{ci_hit[0]:.4f}, {ci_hit[1]:.4f}]")
print(f"Null 95% interval (sample freq): [{ci_freq[0]:.4f}, {ci_freq[1]:.4f}]")

print(f"Significant hit-rate enrichment? {'YES' if p_hit < 0.05 else 'NO'}")
print(f"Significant sample-freq enrichment? {'YES' if p_freq < 0.05 else 'NO'}")

Observed anchor_hit_rate_in_topk_slots: 0.2200
Random mean±std: 0.1002 ± 0.0216
Empirical p-value (hit rate): 5e-05
Observed sample_freq_any_anchor_in_topk: 0.7667
Random mean±std: 0.4469 ± 0.0902
Empirical p-value (sample freq): 0.00025
Z-score (hit rate): 5.54
Z-score (sample freq): 3.54
Null 95% interval (hit rate): [0.0600, 0.1467]
Null 95% interval (sample freq): [0.2667, 0.6333]
Significant hit-rate enrichment? YES
Significant sample-freq enrichment? YES


In [32]:
null_hit_mean = np.nanmean(null_hit_rates)
null_freq_mean = np.nanmean(null_sample_freqs)

p_hit_2s = (
    1
    + np.sum(np.abs(null_hit_rates - null_hit_mean) >= np.abs(obs_hit_rate - null_hit_mean))
) / (n_perm + 1)

p_freq_2s = (
    1
    + np.sum(np.abs(null_sample_freqs - null_freq_mean) >= np.abs(obs_sample_freq - null_freq_mean))
) / (n_perm + 1)

print(f"Empirical two-sided p-value (hit rate): {p_hit_2s:.4g}")
print(f"Empirical two-sided p-value (sample freq): {p_freq_2s:.4g}")

print(f"Significant hit-rate deviation (two-sided)? {'YES' if p_hit_2s < 0.05 else 'NO'}")
print(f"Significant sample-freq deviation (two-sided)? {'YES' if p_freq_2s < 0.05 else 'NO'}")

Empirical two-sided p-value (hit rate): 0.0004998
Empirical two-sided p-value (sample freq): 0.002499
Significant hit-rate deviation (two-sided)? YES
Significant sample-freq deviation (two-sided)? YES


Comparison and significance of other distance metrics (Chamfer distance and RMSD)

In [52]:
# Permutation significance test for top-k ranking similarity (paired by sample)


# Set your two score keys
key_a = "normalized_shapley_values_42"
key_b = "normalized_shapley_values_chamfer_42"  # <-- change
k = 5
n_perm = 2000
rng = np.random.default_rng(42)

def _safe_minmax(x):
    lo, hi = np.nanmin(x), np.nanmax(x)
    if not np.isfinite(lo) or not np.isfinite(hi) or (hi - lo) < 1e-12:
        return np.zeros_like(x, dtype=float)
    return (x - lo) / (hi - lo)

def _dcg(rel):
    denom = np.log2(np.arange(2, rel.size + 2))
    return np.sum((2.0**rel - 1.0) / denom)

def _sym_ndcg_at_k(a, b, kk):
    rel_a = _safe_minmax(a)
    rel_b = _safe_minmax(b)

    ord_b = np.argsort(b)[::-1][:kk]
    ord_a = np.argsort(a)[::-1][:kk]

    dcg_ab = _dcg(rel_a[ord_b]); idcg_a = _dcg(rel_a[ord_a]) + 1e-12
    dcg_ba = _dcg(rel_b[ord_a]); idcg_b = _dcg(rel_b[ord_b]) + 1e-12
    return 0.5 * (dcg_ab / idcg_a + dcg_ba / idcg_b)

# def _topk_jaccard(a, b, kk):
#     ta = set(np.argsort(a)[::-1][:kk])
#     tb = set(np.argsort(b)[::-1][:kk])
#     return len(ta & tb) / max(1, len(ta | tb))

# collect paired samples
pairs = []
for i in range(len(data_list)):
    if key_a not in data_dict[i] or key_b not in data_dict[i]:
        continue
    a = np.asarray(data_dict[i][key_a], dtype=float)
    b = np.asarray(data_dict[i][key_b], dtype=float)
    m = min(a.size, b.size)
    if m < 2:
        continue
    a, b = a[:m], b[:m]
    mask = np.isfinite(a) & np.isfinite(b)
    a, b = a[mask], b[mask]
    if a.size < 2:
        continue
    pairs.append((a, b))

if len(pairs) == 0:
    raise ValueError("No valid paired samples found. Check key_a/key_b.")

# observed mean similarities
obs_j, obs_n = [], []
for a, b in pairs:
    kk = min(k, a.size)
    # obs_j.append(_topk_jaccard(a, b, kk))
    obs_n.append(_sym_ndcg_at_k(a, b, kk))
# obs_j = float(np.mean(obs_j))
obs_n = float(np.mean(obs_n))

# null: break correspondence by permuting item order in b within each sample
# null_j = np.empty(n_perm, dtype=float)
null_n = np.empty(n_perm, dtype=float)

for p in range(n_perm):
    # vals_j, vals_n = [], []
    vals_n = []
    for a, b in pairs:
        kk = min(k, a.size)
        b_perm = b[rng.permutation(b.size)]
        # vals_j.append(_topk_jaccard(a, b_perm, kk))
        vals_n.append(_sym_ndcg_at_k(a, b_perm, kk))
    # null_j[p] = np.mean(vals_j)
    null_n[p] = np.mean(vals_n)

# one-sided p-values: "similarity > chance"
# p_j = (1 + np.sum(null_j >= obs_j)) / (n_perm + 1)
p_n = (1 + np.sum(null_n >= obs_n)) / (n_perm + 1)

print(f"Paired samples used: {len(pairs)}")
# print(f"Observed mean Top-{k} Jaccard: {obs_j:.4f}")
print(f"Observed mean symmetric NDCG@{k}: {obs_n:.4f}")
# print(f"Permutation p-value (Jaccard, one-sided): {p_j:.4g}")
print(f"Permutation p-value (NDCG, one-sided): {p_n:.4g}")
# print(f"Top-{k} Jaccard above chance? {'YES' if p_j < 0.05 else 'NO'}")
print(f"NDCG@{k} above chance? {'YES' if p_n < 0.05 else 'NO'}")

Paired samples used: 30
Observed mean symmetric NDCG@5: 0.9831
Permutation p-value (NDCG, one-sided): 0.0004998
NDCG@5 above chance? YES


More than two lists

In [56]:
# Example: multiple score lists per sample in data_dict
score_keys = [
    "normalized_shapley_values_42",
    "normalized_shapley_values_chamfer_42",
    "normalized_shapley_values_rmsd_42",
]
ndcg_k = 5
n_perm = 2000  # permutation count for aggregated NDCG significance

def _clean_pair(a, b):
    m = min(a.size, b.size)
    if m < 2:
        return None, None
    a, b = a[:m], b[:m]
    mask = np.isfinite(a) & np.isfinite(b)
    a, b = a[mask], b[mask]
    if a.size < 2:
        return None, None
    return a, b

def _safe_minmax(x):
    lo, hi = np.nanmin(x), np.nanmax(x)
    if not np.isfinite(lo) or not np.isfinite(hi) or (hi - lo) < 1e-12:
        return np.zeros_like(x, dtype=float)
    return (x - lo) / (hi - lo)

def _dcg(rel):
    denom = np.log2(np.arange(2, rel.size + 2))
    return np.sum((2.0**rel - 1.0) / denom)

def _sym_ndcg_at_k(a, b, k):
    rel_a = _safe_minmax(a)
    rel_b = _safe_minmax(b)

    ord_b = np.argsort(b)[::-1][:k]
    ord_a = np.argsort(a)[::-1][:k]

    dcg_ab = _dcg(rel_a[ord_b]); idcg_a = _dcg(rel_a[ord_a]) + 1e-12
    dcg_ba = _dcg(rel_b[ord_a]); idcg_b = _dcg(rel_b[ord_b]) + 1e-12
    return float(0.5 * (dcg_ab / idcg_a + dcg_ba / idcg_b))

def _ndcg_perm_test_group(group_pairs, n_perm=2000, rng_obj=None):
    """
    group_pairs: list of tuples (a, b, k_i), one tuple per valid sample.
    Tests whether mean symmetric NDCG across samples is larger than permutation null.
    """
    if rng_obj is None:
        rng_obj = np.random.default_rng(0)

    obs_vals = np.array([_sym_ndcg_at_k(a, b, k_i) for a, b, k_i in group_pairs], dtype=float)
    obs_mean = float(np.mean(obs_vals))

    null_means = np.empty(n_perm, dtype=float)
    for t in range(n_perm):
        perm_vals = []
        for a, b, k_i in group_pairs:
            bp = b[rng_obj.permutation(b.size)]
            perm_vals.append(_sym_ndcg_at_k(a, bp, k_i))
        null_means[t] = float(np.mean(perm_vals))

    p = (1.0 + np.sum(null_means >= obs_mean)) / (n_perm + 1.0)  # one-sided
    mu = float(np.mean(null_means))
    sd = float(np.std(null_means, ddof=1)) + 1e-12
    z = (obs_mean - mu) / sd
    return obs_mean, p, z, obs_vals

# Use existing notebook RNG if available
rng_obj = rng if "rng" in globals() else np.random.default_rng(42)

# -------- 1) Aggregated stats + aggregated permutation test by pair --------
pair_groups = {}  # (A, B) -> list of (a, b, kk)
for i in range(len(data_list)):
    present = [k for k in score_keys if k in data_dict[i]]
    for ka, kb in combinations(present, 2):
        a = np.asarray(data_dict[i][ka], dtype=float)
        b = np.asarray(data_dict[i][kb], dtype=float)
        a, b = _clean_pair(a, b)
        if a is None:
            continue
        kk = min(ndcg_k, a.size)
        pair_groups.setdefault((ka, kb), []).append((a, b, kk))

agg_rows = []
for (ka, kb), group_pairs in pair_groups.items():
    sp = np.array([spearmanr(a, b).correlation for a, b, _ in group_pairs], dtype=float)
    pr = np.array([pearsonr(a, b).statistic for a, b, _ in group_pairs], dtype=float)

    ndcg_mean_obs, ndcg_p, ndcg_z, ndcg_vals = _ndcg_perm_test_group(
        group_pairs, n_perm=n_perm, rng_obj=rng_obj
    )

    agg_rows.append({
        "A": ka,
        "B": kb,
        "n_valid_samples": len(group_pairs),
        "spearman_rho_mean": float(np.nanmean(sp)),
        "spearman_rho_std": float(np.nanstd(sp, ddof=1)) if len(sp) > 1 else np.nan,
        "pearson_r_mean": float(np.nanmean(pr)),
        "pearson_r_std": float(np.nanstd(pr, ddof=1)) if len(pr) > 1 else np.nan,
        "ndcg_at_5_sym_mean": ndcg_mean_obs,
        "ndcg_at_5_sym_std": float(np.nanstd(ndcg_vals, ddof=1)) if len(ndcg_vals) > 1 else np.nan,
        "ndcg_perm_p_agg": ndcg_p,
        "ndcg_perm_z_agg": ndcg_z,
        "ndcg_sig_05_agg": ndcg_p < 0.05,
    })

agg_df = pd.DataFrame(agg_rows).sort_values(["A", "B"]).reset_index(drop=True)

if agg_df.empty:
    print("No valid pairs found.")
else:
    print("Aggregated results across all valid samples per pair:")
    print(agg_df)

# -------- 2) Global rank agreement across >2 lists: Kendall's W --------
def kendalls_w(rank_matrix):
    # rank_matrix shape: (m_raters, n_items)
    m, n = rank_matrix.shape
    R = np.sum(rank_matrix, axis=0)
    R_bar = np.mean(R)
    S = np.sum((R - R_bar) ** 2)
    W = 12 * S / (m**2 * (n**3 - n))
    return float(W)

W_vals = []
for i in range(len(data_list)):
    arrays = []
    for k in score_keys:
        if k not in data_dict[i]:
            arrays = []
            break
        arrays.append(np.asarray(data_dict[i][k], dtype=float))
    if len(arrays) != len(score_keys):
        continue

    n = min(a.size for a in arrays)
    if n < 2:
        continue
    arrays = [a[:n] for a in arrays]

    M = np.vstack(arrays)  # (m_lists, n_items)
    valid = np.all(np.isfinite(M), axis=0)
    M = M[:, valid]
    if M.shape[1] < 2:
        continue

    ranks = np.vstack([rankdata(-row, method="average") for row in M])
    W_vals.append(kendalls_w(ranks))

if len(W_vals) == 0:
    print("No valid samples for Kendall's W.")
else:
    print(f"Mean Kendall's W across samples: {np.mean(W_vals):.4f}")
    print(f"Std Kendall's W across samples: {np.std(W_vals, ddof=1):.4f}")

Aggregated results across all valid samples per pair:
                                      A                                     B  \
0          normalized_shapley_values_42  normalized_shapley_values_chamfer_42   
1          normalized_shapley_values_42     normalized_shapley_values_rmsd_42   
2  normalized_shapley_values_chamfer_42     normalized_shapley_values_rmsd_42   

   n_valid_samples  spearman_rho_mean  spearman_rho_std  pearson_r_mean  \
0               30           0.949237          0.037152        0.971930   
1               30           0.888072          0.097740        0.934175   
2               30           0.911237          0.108851        0.959962   

   pearson_r_std  ndcg_at_5_sym_mean  ndcg_at_5_sym_std  ndcg_perm_p_agg  \
0       0.021125            0.983106           0.020778           0.0005   
1       0.048369            0.960641           0.040184           0.0005   
2       0.036299            0.978714           0.024388           0.0005   

   ndcg_perm_z_

In [55]:
# Aggregated permutation test across all valid sample-pairs (global NDCG agreement)
agg_n_perm = n_perm if "n_perm" in globals() else 2000
agg_rng = np.random.default_rng(42)

# Rebuild valid sample-pairs to make this cell self-contained
agg_pairs = []
for i in range(len(data_list)):
    present = [k for k in score_keys if k in data_dict[i]]
    for ka, kb in combinations(present, 2):
        a = np.asarray(data_dict[i][ka], dtype=float)
        b = np.asarray(data_dict[i][kb], dtype=float)
        a, b = _clean_pair(a, b)
        if a is None:
            continue
        kk = min(ndcg_k, a.size)
        agg_pairs.append((a, b, kk))

if len(agg_pairs) == 0:
    print("No valid pairs found for aggregated test.")
else:
    # Observed global statistic: mean symmetric NDCG across all sample-pairs
    obs_vals = [_sym_ndcg_at_k(a, b, kk) for a, b, kk in agg_pairs]
    agg_obs = float(np.mean(obs_vals))

    # Null distribution: break correspondence within each pair
    agg_null = np.empty(agg_n_perm, dtype=float)
    for p in range(agg_n_perm):
        perm_vals = []
        for a, b, kk in agg_pairs:
            b_perm = b[agg_rng.permutation(b.size)]
            perm_vals.append(_sym_ndcg_at_k(a, b_perm, kk))
        agg_null[p] = float(np.mean(perm_vals))

    # Empirical p-values
    agg_p_one = (1 + np.sum(agg_null >= agg_obs)) / (agg_n_perm + 1)
    agg_mu = float(np.mean(agg_null))
    agg_sd = float(np.std(agg_null, ddof=1)) + 1e-12
    agg_z = (agg_obs - agg_mu) / agg_sd
    agg_ci = np.percentile(agg_null, [2.5, 97.5])
    agg_p_two = (1 + np.sum(np.abs(agg_null - agg_mu) >= np.abs(agg_obs - agg_mu))) / (agg_n_perm + 1)
    agg_p_min = 1.0 / (agg_n_perm + 1.0)

    print(f"Aggregated test over {len(agg_pairs)} sample-pairs")
    print(f"Observed mean symmetric NDCG@{ndcg_k}: {agg_obs:.4f}")
    print(f"Null mean±std: {agg_mu:.4f} ± {agg_sd:.4f}")
    print(f"Null 95% interval: [{agg_ci[0]:.4f}, {agg_ci[1]:.4f}]")
    print(f"Z-score: {agg_z:.2f}")
    print(f"Empirical p-value (one-sided, agreement > chance): {agg_p_one:.4g}")
    print(f"Empirical p-value (two-sided): {agg_p_two:.4g}")
    print(f"Minimum non-zero empirical p with {agg_n_perm} permutations: {agg_p_min:.4g}")

Aggregated test over 90 sample-pairs
Observed mean symmetric NDCG@5: 0.9742
Null mean±std: 0.4496 ± 0.0144
Null 95% interval: [0.4231, 0.4788]
Z-score: 36.39
Empirical p-value (one-sided, agreement > chance): 0.0004998
Empirical p-value (two-sided): 0.0004998
Minimum non-zero empirical p with 2000 permutations: 0.0004998


permutation p-value for Kendall's W

In [57]:
n_perm = 20000
rng = np.random.default_rng(42)

def kendalls_w(rank_matrix):
    m, n = rank_matrix.shape
    if m < 2 or n < 2:
        return np.nan
    R = np.sum(rank_matrix, axis=0)
    S = np.sum((R - np.mean(R)) ** 2)
    return float(12 * S / (m**2 * (n**3 - n)))

# Collect aligned per-sample matrices
sample_mats = []
for i in range(len(data_list)):
    if not all(k in data_dict[i] for k in score_keys):
        continue

    arrays = [np.asarray(data_dict[i][k], dtype=float) for k in score_keys]
    n = min(arr.size for arr in arrays)
    if n < 2:
        continue

    M = np.vstack([arr[:n] for arr in arrays])   # (m_lists, n_items)
    valid = np.all(np.isfinite(M), axis=0)
    M = M[:, valid]
    if M.shape[1] < 2:
        continue

    sample_mats.append(M)

if len(sample_mats) == 0:
    print("No valid samples for permutation test.")
else:
    # Observed W (mean across samples)
    obs_w = []
    for M in sample_mats:
        ranks = np.vstack([rankdata(-row, method="average") for row in M])
        obs_w.append(kendalls_w(ranks))
    obs_w = np.array(obs_w, dtype=float)
    obs_mean = float(np.nanmean(obs_w))

    # Null: destroy cross-list correspondence by permuting items independently per list
    null_mean_w = np.empty(n_perm, dtype=float)
    for p in range(n_perm):
        vals = []
        for M in sample_mats:
            M_perm = np.vstack([row[rng.permutation(row.size)] for row in M])
            ranks_p = np.vstack([rankdata(-row, method="average") for row in M_perm])
            vals.append(kendalls_w(ranks_p))
        null_mean_w[p] = np.nanmean(vals)

    # p-values
    p_one_sided = (1 + np.sum(null_mean_w >= obs_mean)) / (n_perm + 1)  # agreement > chance
    null_center = np.nanmean(null_mean_w)
    p_two_sided = (1 + np.sum(np.abs(null_mean_w - null_center) >= np.abs(obs_mean - null_center))) / (n_perm + 1)

    ci = np.nanpercentile(null_mean_w, [2.5, 97.5])
    z = (obs_mean - np.nanmean(null_mean_w)) / (np.nanstd(null_mean_w, ddof=1) + 1e-12)

    print(f"Valid samples: {len(sample_mats)}")
    print(f"Observed mean Kendall's W: {obs_mean:.4f}")
    print(f"Null mean±std: {np.nanmean(null_mean_w):.4f} ± {np.nanstd(null_mean_w, ddof=1):.4f}")
    print(f"Null 95% interval: [{ci[0]:.4f}, {ci[1]:.4f}]")
    print(f"Z-score: {z:.2f}")
    print(f"Permutation p-value (one-sided, W > chance): {p_one_sided:.4g}")
    print(f"Permutation p-value (two-sided): {p_two_sided:.4g}")

Valid samples: 30
Observed mean Kendall's W: 0.9441
Null mean±std: 0.3335 ± 0.0162
Null 95% interval: [0.3021, 0.3659]
Z-score: 37.65
Permutation p-value (one-sided, W > chance): 5e-05
Permutation p-value (two-sided): 5e-05


All metrics

In [58]:
n_perm = 2000
rng = np.random.default_rng(42)

# Use existing notebook vars if present
ndcg_k = ndcg_k if "ndcg_k" in globals() else 5

def kendalls_w(rank_matrix):
    m, n = rank_matrix.shape
    if m < 2 or n < 2:
        return np.nan
    R = np.sum(rank_matrix, axis=0)
    S = np.sum((R - np.mean(R)) ** 2)
    return float(12 * S / (m**2 * (n**3 - n)))

def _safe_minmax(x):
    lo, hi = np.nanmin(x), np.nanmax(x)
    if not np.isfinite(lo) or not np.isfinite(hi) or (hi - lo) < 1e-12:
        return np.zeros_like(x, dtype=float)
    return (x - lo) / (hi - lo)

def _dcg(rel):
    denom = np.log2(np.arange(2, rel.size + 2))
    return np.sum((2.0**rel - 1.0) / denom)

def _sym_ndcg_at_k(a, b, k):
    rel_a = _safe_minmax(a)
    rel_b = _safe_minmax(b)

    ord_b = np.argsort(b)[::-1][:k]
    ord_a = np.argsort(a)[::-1][:k]

    dcg_ab = _dcg(rel_a[ord_b]); idcg_a = _dcg(rel_a[ord_a]) + 1e-12
    dcg_ba = _dcg(rel_b[ord_a]); idcg_b = _dcg(rel_b[ord_b]) + 1e-12
    return float(0.5 * (dcg_ab / idcg_a + dcg_ba / idcg_b))

# Collect aligned per-sample matrices
sample_mats = []
for i in range(len(data_list)):
    if not all(k in data_dict[i] for k in score_keys):
        continue

    arrays = [np.asarray(data_dict[i][k], dtype=float) for k in score_keys]
    n = min(arr.size for arr in arrays)
    if n < 2:
        continue

    M = np.vstack([arr[:n] for arr in arrays])   # (m_lists, n_items)
    valid = np.all(np.isfinite(M), axis=0)
    M = M[:, valid]
    if M.shape[1] < 2:
        continue

    sample_mats.append(M)

if len(sample_mats) == 0:
    print("No valid samples for permutation test.")
else:
    metric_names = [
        "spearman_rho",
        "kendall_tau",
        "pearson_r",
        "ndcg_sym",
        "kendalls_w",
    ]

    def compute_sample_metrics(M):
        m_lists, n_items = M.shape
        pair_vals = {k: [] for k in metric_names if k != "kendalls_w"}

        for a in range(m_lists):
            for b in range(a + 1, m_lists):
                x, y = M[a], M[b]
                pair_vals["spearman_rho"].append(spearmanr(x, y).correlation)
                pair_vals["kendall_tau"].append(kendalltau(x, y).correlation)
                pair_vals["pearson_r"].append(pearsonr(x, y).statistic)
                kk = min(ndcg_k, n_items)
                pair_vals["ndcg_sym"].append(_sym_ndcg_at_k(x, y, kk))

        ranks = np.vstack([rankdata(-row, method="average") for row in M])
        out = {k: float(np.nanmean(v)) for k, v in pair_vals.items()}
        out["kendalls_w"] = kendalls_w(ranks)
        return out

    # Observed
    obs_per_sample = [compute_sample_metrics(M) for M in sample_mats]
    obs_mean = {k: float(np.nanmean([d[k] for d in obs_per_sample])) for k in metric_names}

    # Null distribution (destroy cross-list correspondence)
    null = {k: np.empty(n_perm, dtype=float) for k in metric_names}
    for p in range(n_perm):
        vals = {k: [] for k in metric_names}
        for M in sample_mats:
            M_perm = np.vstack([row[rng.permutation(row.size)] for row in M])
            d = compute_sample_metrics(M_perm)
            for k in metric_names:
                vals[k].append(d[k])
        for k in metric_names:
            null[k][p] = float(np.nanmean(vals[k]))

    # p-values and effect sizes
    higher_better = {"spearman_rho", "kendall_tau", "pearson_r", "ndcg_sym", "kendalls_w"}

    print(f"Valid samples: {len(sample_mats)}")
    for k in metric_names:
        obs = obs_mean[k]
        null_mu = float(np.nanmean(null[k]))
        null_sd = float(np.nanstd(null[k], ddof=1))
        ci = np.nanpercentile(null[k], [2.5, 97.5])

        p_one = (1 + np.sum(null[k] >= obs)) / (n_perm + 1)
        z = (obs - null_mu) / (null_sd + 1e-12)
        p_two = (1 + np.sum(np.abs(null[k] - null_mu) >= np.abs(obs - null_mu))) / (n_perm + 1)

        print(f"\n{k}:")
        print(f"  observed mean: {obs:.4f}")
        print(f"  null mean±std: {null_mu:.4f} ± {null_sd:.4f}")
        print(f"  null 95% CI: [{ci[0]:.4f}, {ci[1]:.4f}]")
        print(f"  z-score: {z:.2f}")
        print(f"  p(one-sided): {p_one:.4g}")
        print(f"  p(two-sided): {p_two:.4g}")


Valid samples: 30

spearman_rho:
  observed mean: 0.9162
  null mean±std: 0.0006 ± 0.0238
  null 95% CI: [-0.0457, 0.0483]
  z-score: 38.53
  p(one-sided): 0.0004998
  p(two-sided): 0.0004998

kendall_tau:
  observed mean: 0.8041
  null mean±std: 0.0004 ± 0.0168
  null 95% CI: [-0.0330, 0.0337]
  z-score: 47.92
  p(one-sided): 0.0004998
  p(two-sided): 0.0004998

pearson_r:
  observed mean: 0.9554
  null mean±std: 0.0008 ± 0.0239
  null 95% CI: [-0.0449, 0.0492]
  z-score: 40.00
  p(one-sided): 0.0004998
  p(two-sided): 0.0004998

ndcg_sym:
  observed mean: 0.9742
  null mean±std: 0.4501 ± 0.0143
  null 95% CI: [0.4222, 0.4795]
  z-score: 36.73
  p(one-sided): 0.0004998
  p(two-sided): 0.0004998

kendalls_w:
  observed mean: 0.9441
  null mean±std: 0.3338 ± 0.0158
  null 95% CI: [0.3029, 0.3655]
  z-score: 38.53
  p(one-sided): 0.0004998
  p(two-sided): 0.0004998
